In [0]:
# Environment variables
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate() 

In [0]:
# Load the CSV file into a DataFrame from Databricks Catalog
catalog_name = "10alytics_netflex_workspace"
schema_name = "default"
volume_name = "dataset"
file_name = "Netflix_Movies_and_TV_Shows.csv"

file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{file_name}"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

display(df)

In [ ]:
# Load the CSV file into a DataFrame from Azure Data Lake Storage (ADLS)
file_path_adls = (
    "abfss://esonetlixdata@esonetflixstorageaccount.dfs.core.windows.net/raw/Netflix_Movies_and_TV_Shows.csv"
)
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path_adls)

display(df)

In [0]:
df = df.toDF(*[
    c.replace(' ', '_') for c in df.columns
])

In [0]:
# Create table view to use sql to transform data
df.createOrReplaceTempView('movies')

In [0]:
%sql 
-- Use sql to transform data
select * from movies

In [0]:
%sql
select title, type, genre, release_year, rating, duration,
    country, concat(left(card_number, 4), '****_****') as card_number_masked
from movies

In [0]:
# Build the transformed DataFrame. The opening parenthesis must be on the
# same logical line as spark.sql; otherwise transformed_df becomes a method.
transformed_df = spark.sql(
    """
    SELECT
        title,
        type,
        genre,
        release_year,
        rating,
        duration,
        country,
        concat(left(cast(card_number AS string), 4), '****_****') AS card_number_masked
    FROM movies
    """
)
display(transformed_df)

In [0]:
catalog_name = "10alytics_netflex_workspace"
schema_name = "default"
volume_name = "dataset"
# Spark writes CSV output as a directory containing part files.
output_directory = "processed/netflix_movies"

output_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{output_directory}"

In [0]:
# Save to ADLS Gen2
output_path_adls = "abfss://esonetlixdata@esonetflixstorageaccount.dfs.core.windows.net/movies"

In [0]:
# Write the transformed DataFrame to the configured output directory.
(
    transformed_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(output_path_adls)
)

In [0]:
print(f"CSV output written to: {output_path}")
print(f"CSV output written to: {output_path_adls}")